# Step 1a: RTMS 매매 실거래가 수집**목적**: 국토부 실거래가 API(RTMS)에서 수도권 66개 시군구의 매매 데이터를 월 단위로 수집해 raw 디렉토리에 보존.**갱신 주기**: 일 1회 (새벽 04:00 cron). 직전 90일까지 재수집해서 정정·해제 반영.**핵심 함정** (설계서 1.1절):1. 같은 거래가 여러 번 들어올 수 있음 — 신고일 기준이라 정정 시 새 행으로 추가됨2. 해제(`해제여부=Y`)는 영상 송출 전 반드시 필터3. XML 응답이 안정적 (JSON 옵션은 가끔 인코딩 깨짐)4. 호출 한도 일 10,000건 — 수도권 66개 × 한 달 = 약 66회 호출, 충분

## 1. 환경 로드

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    sys.path.insert(0, '/content/drive/MyDrive/realestate_reco')
except ImportError:
    pass  # 로컬 환경

from common import *
import time
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import pandas as pd

SECRETS = load_secrets()
SERVICE_KEY = SECRETS['data_go_kr_service_key']
MOCK_MODE = SECRETS.get('MOCK_MODE', True)
print(f'MOCK_MODE = {MOCK_MODE}')
print(f'시군구 수 = {len(SIGUNGU_CODES)}')

## 2. RTMS API 호출 함수**엔드포인트**: `getRTMSDataSvcAptTradeDev` (개발 버전, 안정적)**파라미터**: `LAWD_CD` (시군구 5자리), `DEAL_YMD` (계약 연월 6자리)**응답**: XML, 한 호출에 최대 1000건 (페이지네이션 필요 시 numOfRows·pageNo)

In [ ]:
RTMS_TRADE_URL = 'http://openapi.molit.go.kr/OpenAPI_ToolInstallPackage/service/rest/RTMSOBJSvc/getRTMSDataSvcAptTradeDev'def fetch_rtms_trade(sigungu_code: str, ym: str,                     num_of_rows: int = 1000,                     page_no: int = 1,                     timeout: int = 30) -> str:    """RTMS 매매 API 호출. 원본 XML 텍스트 반환.    sigungu_code: '11680' (강남구)    ym: '202604' (2026년 4월)    """    if MOCK_MODE:        return _mock_rtms_response(sigungu_code, ym, num_of_rows)    params = {        'serviceKey': SERVICE_KEY,        'LAWD_CD': sigungu_code,        'DEAL_YMD': ym,        'numOfRows': num_of_rows,        'pageNo': page_no,    }    resp = requests.get(RTMS_TRADE_URL, params=params, timeout=timeout)    resp.raise_for_status()    return resp.textdef _mock_rtms_response(sigungu_code, ym, n):    """MOCK_MODE: 가상 거래 5건 생성. 파이프라인 동작 확인용."""    items_xml = ''    sigungu_name = SIGUNGU_CODES.get(sigungu_code, '미지정')    for i in range(5):        items_xml += f'''    <item>      <거래금액>{120000 + i*5000:>10,}</거래금액>      <건축년도>{2010 + i}</건축년도>      <년>{ym[:4]}</년>      <법정동> {sigungu_name} 테스트동</법정동>      <아파트>테스트단지{i+1}차</아파트>      <월>{int(ym[4:])}</월>      <일>{15 + i}</일>      <전용면적>{84.5 + i*5.0}</전용면적>      <지번>{100+i}</지번>      <지역코드>{sigungu_code}</지역코드>      <층>{5 + i*2}</층>      <도로명>테스트로</도로명>      <도로명건물본번호코드>{i+1}</도로명건물본번호코드>      <해제여부></해제여부>      <해제사유발생일></해제사유발생일>    </item>'''    return f'''<?xml version="1.0" encoding="UTF-8"?><response><header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg></header><body><items>{items_xml}</items><numOfRows>{n}</numOfRows><pageNo>1</pageNo><totalCount>5</totalCount></body></response>'''# 테스트 호출xml_text = fetch_rtms_trade('11680', '202604')print(xml_text[:500] + '...')

## 3. XML 파싱 함수응답을 DataFrame으로 변환. 모든 필드는 일단 문자열로 받고 다음 단계(Step 2 정제)에서 타입 캐스팅.

In [ ]:
def parse_rtms_xml(xml_text: str) -> pd.DataFrame:    """RTMS 응답 XML → DataFrame.    raw 단계에서는 타입 변환을 최소화한다 (원본 충실 보존).    """    root = ET.fromstring(xml_text)    # 응답 코드 체크    code_elem = root.find('.//resultCode')    if code_elem is not None and code_elem.text != '00':        msg = root.find('.//resultMsg')        msg_text = msg.text if msg is not None else 'unknown'        raise RuntimeError(f'RTMS API error: {code_elem.text} - {msg_text}')    items = root.findall('.//item')    rows = []    for item in items:        row = {child.tag: (child.text or '').strip() for child in item}        rows.append(row)    if not rows:        return pd.DataFrame()    df = pd.DataFrame(rows)    return df# 테스트df_test = parse_rtms_xml(xml_text)print(f'행수: {len(df_test)}')print(f'컬럼: {list(df_test.columns)}')df_test.head()

## 4. 시군구·월 단위 수집 함수raw 디렉토리에 XML 원본 그대로 저장 + .meta.json 동봉.**멱등성**: 같은 (sigungu, ym)을 두 번 호출해도 파일이 덮어써지므로 안전.

In [ ]:
def collect_one_month(sigungu_code: str, ym: str,                      retries: int = 3,                      sleep_between: float = 0.3) -> dict:    """한 시군구·한 달 매매 데이터 수집.    Returns: {'rows': N, 'path': raw_xml_path, 'status': 'OK'/'EMPTY'/'ERROR'}    """    year, month = ym[:4], ym[4:]    out_dir = DIRS['raw_trade'] / year / month    out_dir.mkdir(parents=True, exist_ok=True)    xml_path = out_dir / f'{sigungu_code}_{ym}.xml'    meta_path = out_dir / f'{sigungu_code}_{ym}.meta.json'    last_err = None    for attempt in range(1, retries + 1):        try:            xml_text = fetch_rtms_trade(sigungu_code, ym)            df = parse_rtms_xml(xml_text)            xml_path.write_text(xml_text, encoding='utf-8')            meta = {                'sigungu_code': sigungu_code,                'sigungu_name': SIGUNGU_CODES.get(sigungu_code),                'ym': ym,                'fetched_at': datetime.now().isoformat(),                'rows': len(df),                'attempt': attempt,                'mock': MOCK_MODE,            }            meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')            time.sleep(sleep_between)            return {'rows': len(df), 'path': str(xml_path),                    'status': 'EMPTY' if len(df) == 0 else 'OK'}        except Exception as e:            last_err = e            time.sleep(2 ** attempt)  # exponential backoff: 2s, 4s, 8s    # 재시도 모두 실패    return {'rows': 0, 'path': None, 'status': 'ERROR', 'error': str(last_err)}# 단일 시군구·월 테스트result = collect_one_month('11680', '202604')print(result)

## 5. 수도권 전체·한 달 수집66개 시군구를 순회. MOCK_MODE 아니면 약 5분 소요.**주의**: 새벽 cron으로 돌릴 때는 직전 3개월(현재 월 + 직전 2개월)을 재수집한다.정정·해제 신고가 D+90까지 들어오기 때문.

In [ ]:
def collect_metro_one_month(ym: str, verbose: bool = True) -> pd.DataFrame:    """수도권 66개 시군구 한 달 매매 수집.    각 시군구 결과 요약을 DataFrame으로 반환 — 운영 모니터링용.    """    results = []    for code, name in SIGUNGU_CODES.items():        r = collect_one_month(code, ym)        r['sigungu_code'] = code        r['sigungu_name'] = name        r['ym'] = ym        results.append(r)        if verbose:            print(f"  {code} {name:14s} {r['status']:6s} {r['rows']:4d}건")    summary = pd.DataFrame(results)    return summary# 직전 1개월 수집 (테스트)target_ym = '202604'summary = collect_metro_one_month(target_ym, verbose=False)print(f"\n=== {target_ym} 수집 요약 ===")print(f'OK:    {(summary.status == "OK").sum():3d}개')print(f'EMPTY: {(summary.status == "EMPTY").sum():3d}개')print(f'ERROR: {(summary.status == "ERROR").sum():3d}개')print(f'총 행수: {summary.rows.sum():,}건')if (summary.status == 'ERROR').any():    print('\n에러 시군구:')    print(summary[summary.status == 'ERROR'][['sigungu_code', 'sigungu_name', 'error']].to_string())

## 6. 직전 90일 재수집 (정정·해제 반영)cron 매일 04:00 호출용 메인 함수.

In [ ]:
def daily_collect_recent(today: datetime = None) -> pd.DataFrame:    """오늘 날짜 기준 직전 3개월 (정확히는 현재월 + 직전 2개월)을 재수집.    이렇게 해야 정정·해제 신고를 놓치지 않는다.    """    today = today or datetime.now()    yms = []    for offset in range(2, -1, -1):  # 2개월 전, 1개월 전, 이번 달        d = today - timedelta(days=30 * offset)        yms.append(f'{d.year:04d}{d.month:02d}')    yms = list(dict.fromkeys(yms))  # 중복 제거 유지 순서    print(f'재수집 대상 월: {yms}')    all_summaries = []    for ym in yms:        print(f'\n=== {ym} 수집 시작 ===')        s = collect_metro_one_month(ym, verbose=False)        all_summaries.append(s)    return pd.concat(all_summaries, ignore_index=True)# 운영 시 매일 새벽 호출# summary = daily_collect_recent()

## 7. 수집 결과 로그 저장

In [ ]:
def save_ingest_log(summary: pd.DataFrame, job_name: str = 'ingest_trade'):    """수집 결과를 logs/에 일자별로 보존."""    today = datetime.now().strftime('%Y-%m-%d')    log_path = DIRS['logs'] / f'{job_name}_{today}.json'    payload = {        'job': job_name,        'date': today,        'total_rows': int(summary.rows.sum()),        'ok_count': int((summary.status == 'OK').sum()),        'empty_count': int((summary.status == 'EMPTY').sum()),        'error_count': int((summary.status == 'ERROR').sum()),        'errors': summary[summary.status == 'ERROR'].to_dict('records'),    }    log_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')    print(f'로그 저장: {log_path}')    # P1 알림 트리거 (설계서 4.6절)    if payload['error_count'] > 0:        print(f"⚠ ERROR {payload['error_count']}건 — Slack 알림 보내야 함")    return payloadsave_ingest_log(summary)

---## 다음 단계- **Step 1b**: 단지정보(K-apt) 월 스냅샷 수집- **Step 2**: raw XML → staged Parquet 정제 + apt_id 부여- **Step 3**: 통합 master 테이블 빌드